# 💰 Cost Optimization Patterns

## Learning Objectives
In this notebook, you will learn:
1. **Model Routing** - How to send simple queries to a cheap model and complex queries to a more capable (and expensive) one
2. **Semantic Caching** - How to avoid paying for the same LLM call twice by caching responses
3. **Token Budgeting** - How to track token usage and enforce a hard cap per request
4. **Cost Estimation** - How to estimate the dollar cost of an LLM call from token counts

## Prerequisites
- Basic understanding of LangChain and `ChatOpenAI`
- Familiarity with Python classes and decorators
- An `OPENAI_API_KEY` in your `.env` file
- (Optional) A LangSmith API key for the `@traceable` tracing decorator to log runs

---
## 🔧 Part 1: Environment Setup

We start by importing the libraries used throughout this notebook and loading environment
variables from a local `.env` file (via `python-dotenv`) rather than hardcoding API keys.

In [ ]:
# ============================================================================
# SETUP: Imports and Environment Configuration
# ============================================================================
import hashlib
from functools import lru_cache
from typing import Callable, Optional

from dotenv import load_dotenv
from langsmith import traceable

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

load_dotenv()

print("✅ Environment loaded and imports ready!")

---
## 🚦 Part 2: Model Routing — Route by Query Complexity

The single biggest cost lever in production LLM systems is **not sending every query to your
most expensive model**. A model router classifies each incoming query as "simple" or "complex"
and dispatches it to a cheap model (e.g. `gpt-4o-mini`) or a stronger, pricier one (e.g. `gpt-4o`)
accordingly.

### Key Concepts:
- **Classifier model**: A cheap model used just to label query complexity, so the classification
  itself stays inexpensive
- **Cost-per-1k tokens**: The per-model input pricing used to estimate spend for each call

### `ModelRouter`
Classifies each query's complexity, then routes it to the cheap or expensive model and
returns the response along with the model used and an estimated cost.

In [ ]:
# ============================================================================
# MODEL ROUTER: Route Queries to a Cheap or Expensive Model Based on Complexity
# ============================================================================
class ModelRouter:
    """Route queries to appropriate model based on complexity."""

    def __init__(self):
        self.cheap_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        self.expensive_model = ChatOpenAI(model="gpt-4o", temperature=0)
        self.classifier = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    def classify_complexity(self, query: str) -> str:
        """Classify query complexity."""

        prompt = ChatPromptTemplate.from_template(
            """
Classify this query's complexity as 'simple' or 'complex'.

Simple: Basic facts, short answers, simple calculations
Complex: Analysis, reasoning, creative tasks, multi-step problems

Query: {query}

Respond with only: simple or complex
"""
        )

        response = self.classifier.invoke(prompt.format(query=query))
        return response.content.strip().lower()

    @traceable(name="routed_query")
    def invoke(self, query: str) -> tuple[str, str, float]:
        """
        Route and invoke query.
        Returns: (response, model_used, estimated_cost)
        """
        complexity = self.classify_complexity(query)

        if complexity == "simple":
            model = self.cheap_model
            model_name = "gpt-4o-mini"
            cost_per_1k = 0.00015  # Input cost
        else:
            model = self.expensive_model
            model_name = "gpt-4o"
            cost_per_1k = 0.0025  # Input cost

        response = model.invoke(query)

        # Estimate cost (rough)
        tokens = len(query.split()) * 1.3  # Rough token estimate
        estimated_cost = (tokens / 1000) * cost_per_1k

        return response.content, model_name, estimated_cost

### `demo_model_routing`
Runs a few sample queries of varying complexity through the router and prints which model
handled each one along with its estimated cost.

In [ ]:
# ============================================================================
# DEMO: Model Routing Example
# ============================================================================
def demo_model_routing():
    """Demonstrate model routing."""

    router = ModelRouter()

    queries = [
        "What is 2 + 2?",  # Simple
        "Analyze the economic implications of AI on the job market.",  # Complex
        "What color is the sky?",  # Simple
    ]

    print("Model Routing Demo:\n")

    total_cost = 0
    for query in queries:
        result, model, cost = router.invoke(query)
        total_cost += cost
        print(f"Query: {query[:50]}...")
        print(f"  Model: {model}")
        print(f"  Est. Cost: ${cost:.6f}")
        print(f"  Response: {result[:50]}...")

    print(f"\nTotal Estimated Cost: ${total_cost:.6f}")

---
## 🧠 Part 3: Semantic Caching — Avoid Redundant LLM Calls

If the same (or a normalized-equivalent) query comes in twice, there is no reason to pay for
the LLM call a second time. A cache keyed on a normalized hash of the query text can serve
repeat queries for free.

> **Note**: This implementation uses exact-match caching on a normalized query hash. A
> production system would extend `get()` with embedding-based similarity search to also catch
> paraphrased queries, not just identical ones.

### `SemanticCache`
Stores query/response pairs keyed by an MD5 hash of the normalized (lowercased, stripped)
query text, and looks up cached responses on exact match.

In [ ]:
# ============================================================================
# SEMANTIC CACHE: Exact-Match Query Caching
# ============================================================================
class SemanticCache:
    """Cache responses with semantic similarity matching."""

    def __init__(self, similarity_threshold: float = 0.9):
        self.cache = {}
        self.threshold = similarity_threshold
        self.embedder = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    def _hash_query(self, query: str) -> str:
        """Create hash of normalized query."""
        normalized = query.lower().strip()
        return hashlib.md5(normalized.encode()).hexdigest()

    def get(self, query: str) -> Optional[str]:
        """Get cached response if similar query exists."""
        query_hash = self._hash_query(query)

        # Exact match
        if query_hash in self.cache:
            return self.cache[query_hash]["response"]

        # Could add embedding-based similarity here
        # For demo, just use exact match

        return None

    def set(self, query: str, response: str):
        """Cache a response."""
        query_hash = self._hash_query(query)
        self.cache[query_hash] = {"query": query, "response": response}

    def stats(self) -> dict:
        return {"cached_queries": len(self.cache)}

### `CachedLLM`
Wraps an LLM so that every `invoke()` first checks the semantic cache; on a miss it calls the
LLM and stores the result, and it tracks hit/miss counters along the way.

In [ ]:
# ============================================================================
# CACHED LLM: LLM Wrapper with Caching
# ============================================================================
class CachedLLM:
    """LLM wrapper with caching."""

    def __init__(self):
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        self.cache = SemanticCache()
        self.cache_hits = 0
        self.cache_misses = 0

    @traceable(name="cached_invoke")
    def invoke(self, query: str) -> tuple[str, bool]:
        """
        Invoke with caching.
        Returns: (response, from_cache)
        """
        # Check cache
        cached = self.cache.get(query)
        if cached:
            self.cache_hits += 1
            return cached, True

        # Call LLM
        self.cache_misses += 1
        response = self.llm.invoke(query)
        result = response.content

        # Cache result
        self.cache.set(query, result)

        return result, False

    def get_stats(self) -> dict:
        total = self.cache_hits + self.cache_misses
        hit_rate = self.cache_hits / total if total > 0 else 0
        return {
            "hits": self.cache_hits,
            "misses": self.cache_misses,
            "hit_rate": f"{hit_rate:.1%}",
        }

### `demo_caching`
Sends a mix of new, repeated, and case-varied queries through `CachedLLM` to show which ones
hit the cache versus call the LLM.

In [ ]:
# ============================================================================
# DEMO: Caching Example
# ============================================================================
def demo_caching():
    """Demonstrate caching."""

    llm = CachedLLM()

    queries = [
        "What is Python?",
        "What is JavaScript?",
        "What is Python?",  # Cache hit
        "What is python?",  # Cache hit (normalized)
        "What is Rust?",
    ]

    print("\nCaching Demo:\n")

    for query in queries:
        result, from_cache = llm.invoke(query)
        source = "CACHE" if from_cache else "LLM"
        print(f"[{source}] {query} -> {result[:30]}...")

    print(f"\nStats: {llm.get_stats()}")

---
## 📊 Part 4: Token Budgeting — Enforce Per-Request Limits

Even with routing and caching, an unbounded query can still blow through your cost envelope in
a single call. A token budget rejects any request whose estimated token count exceeds a
configured cap, and tracks cumulative usage across all requests.

### Key Concepts:
- **Rough token estimation**: `len(text.split()) * 1.3` approximates token count without needing
  a real tokenizer like `tiktoken`
- **Hard cap enforcement**: Requests exceeding the budget raise a `ValueError` rather than
  silently running (and billing)

### `TokenBudget`
Tracks input/output token usage across requests and checks whether a given piece of text is
within the configured per-request token limit.

In [ ]:
# ============================================================================
# TOKEN BUDGET: Track and Limit Token Usage
# ============================================================================
class TokenBudget:
    """Track and limit token usage."""

    def __init__(self, max_tokens_per_request: int = 4000):
        self.max_per_request = max_tokens_per_request
        self.usage = {"total_input": 0, "total_output": 0, "requests": 0}

    def estimate_tokens(self, text: str) -> int:
        """Rough token estimation (actual would use tiktoken)."""
        return int(len(text.split()) * 1.3)

    def check_budget(self, text: str) -> tuple[bool, int]:
        """Check if request is within budget."""
        tokens = self.estimate_tokens(text)
        return tokens <= self.max_per_request, tokens

    def record_usage(self, input_tokens: int, output_tokens: int):
        """Record token usage."""
        self.usage["total_input"] += input_tokens
        self.usage["total_output"] += output_tokens
        self.usage["requests"] += 1

    def get_stats(self) -> dict:
        return {
            **self.usage,
            "total_tokens": self.usage["total_input"] + self.usage["total_output"],
            "avg_per_request": (
                (self.usage["total_input"] + self.usage["total_output"])
                / max(self.usage["requests"], 1)
            ),
        }

### `BudgetedLLM`
Wraps an LLM so that every `invoke()` first checks the query against the token budget, raising
before making a paid call if the request is over budget, and records usage after each call.

In [ ]:
# ============================================================================
# BUDGETED LLM: LLM with Token Budget Enforcement
# ============================================================================
class BudgetedLLM:
    """LLM with token budgeting."""

    def __init__(self, max_tokens: int = 4000):
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        self.budget = TokenBudget(max_tokens_per_request=max_tokens)

    @traceable(name="budgeted_invoke")
    def invoke(self, query: str) -> str:
        # Check budget
        within_budget, tokens = self.budget.check_budget(query)

        if not within_budget:
            raise ValueError(
                f"Query exceeds token budget: {tokens} > {self.budget.max_per_request}"
            )

        # Execute
        response = self.llm.invoke(query)
        result = response.content

        # Record usage
        output_tokens = self.budget.estimate_tokens(result)
        self.budget.record_usage(tokens, output_tokens)

        return result

    def get_stats(self) -> dict:
        return self.budget.get_stats()

### `demo_token_budgeting`
Runs one query within budget and one deliberately oversized query to show both the successful
path and the `ValueError` raised when a request is rejected.

In [ ]:
# ============================================================================
# DEMO: Token Budgeting Example
# ============================================================================
def demo_token_budgeting():
    """Demonstrate token budgeting."""

    llm = BudgetedLLM(max_tokens=100)

    queries = [
        "What is AI?",  # Within budget
        "Explain " + "very " * 100 + "complex topic",  # Over budget
    ]

    print("\nToken Budgeting Demo:\n")

    for query in queries:
        try:
            result = llm.invoke(query)
            print(f"✅ {query[:40]}... -> {result[:30]}...")
        except ValueError as e:
            print(f"❌ {query[:40]}... -> {e}")

    print(f"\nUsage: {llm.get_stats()}")

---
## ▶️ Part 5: Run the Demos

The original `__main__` guard is kept verbatim below. Jupyter sets `__name__` to `"__main__"`,
so this cell runs as-is — uncomment a different line to run that demo instead.

In [ ]:
# ============================================================================
# RUN: Execute a Demo
# ============================================================================
if __name__ == "__main__":
    # demo_model_routing()
    # demo_caching()
    demo_token_budgeting()

---
## 📝 Summary

In this notebook, we learned:

### 1. Model Routing
- **`ModelRouter`**: Classifies query complexity with a cheap classifier model, then dispatches
  to a cheap or expensive model accordingly
- **`demo_model_routing()`**: Shows routing decisions and per-query cost estimates

### 2. Semantic Caching
- **`SemanticCache`**: Caches responses by a hash of the normalized query text
- **`CachedLLM`**: Wraps an LLM with cache-first lookups and hit/miss tracking
- **`demo_caching()`**: Demonstrates cache hits on repeated and case-varied queries

### 3. Token Budgeting
- **`TokenBudget`**: Tracks cumulative token usage and enforces a per-request cap
- **`BudgetedLLM`**: Wraps an LLM to reject over-budget requests before they are billed
- **`demo_token_budgeting()`**: Shows both a within-budget call and a rejected over-budget call

### Next Steps
- Combine routing, caching, and budgeting into a single production LLM gateway
- Replace the rough word-count token estimate with a real tokenizer (e.g. `tiktoken`)
- Extend `SemanticCache` with embedding-based similarity matching for paraphrased queries